In [1]:
import numpy as np
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
from matplotlib import pyplot as plt
import toml
from pathlib import Path
import os
import glob

## Get Images

In [ ]:
def contains_target(frames, filenames, coords:SkyCoord, fk4_coords=None, n_cutoff=3, arcsec_threshold=100):

    n_target_frames = 0

    for i, frame in enumerate(frames):

        conditions = [
            frame[0].header["CAMNAME"] == "narrow",
            frame[0].header["GRSNAME"] == "clear",
            not (
                "SLITNAME" in frame and (
                    "vortex" in frame[0].header["SLITNAME"] or
                    "corona" in frame[0].header["SLITNAME"]
                )
            )
        ]

        if not np.all(conditions):
            print(f'[Warning]: Skipping file {filenames[i]} due to conditions')
            continue

        if (type(frame[0].header["RA"]) != str) or (type(frame[0].header["DEC"]) != str):
            print('[Warning]: Skipping file $(filenames[i]) due to missing RA/DEC')
            continue

        if frame[0].header["RADECSYS"] == "FK4":
            print(f'[Info]: Using FK4 coordinates for {filenames[i]}')
            if fk4_coords != None:
                ra, dec = fk4_coords
            else:
                print("[Warning]: Files are in FK4 but no Fk4 coordinates provided!")
        
        radec_distance = coords.separation(SkyCoord(frame[0].header["RA"], frame[0].header["DEC"], unit=(u.hourangle, u.deg))).degree * 3600
        
        print(f'[Info]: Coordinates: object={frame[0].header["OBJECT"]} target={frame[0].header["TARGNAME"]} ra={ra} dec={dec} frame_ra={frame[0].header["RA"]} frame_dec={frame[0].header["DEC"]} radec_distance={radec_distance} radecsys={frame[0].header["RADECSYS"]}')

        if radec_distance < arcsec_threshold and frame[0].header["SHRNAME"].lower() == "open":
            print(f'RADEC FRAME filename={filenames[i]} object={frame[0].header["OBJECT"]} targname={frame[0].header["TARGNAME"]} radec_distance={radec_distance}')
            n_target_frames += 1

    if n_target_frames <= n_cutoff:
        print(f'[Error]: Not enough science frames found (only found {n_target_frames}), skipping obslog generation')
        return False
    else: 
        return True

## Sort Frames

In [104]:
def sort_frames(frames, filenames, lampoff_threshold=100.0, arcsec_threshold=100.0, filter_names=None):

    flat_el = 45.0 # degrees, elevation of the flat field frames

    sci = []
    flats = []
    flats_sky = []
    flats_lampon = []
    flats_lampoff = []
    darks = []

    for i, frame in enumerate(frames):

        if filter_names != None:
            if (filter_names in frame[0].header["OBJECT"]):
                print('[Info]: SKIPPED DUE TO FILTER NAME')
                continue

        p1 = SkyCoord(0, flat_el, unit=u.deg)
        p2 = SkyCoord(0, frame[0].header["EL"], unit=u.deg)
        altaz_distance = p2.separation(p1).degree * 3600

        if (altaz_distance <  arcsec_threshold) and (frame[0].header["WCDMSTAT"].lower() == "open" or frame[0].header["WCDMSTAT"].lower() == "idle") and (frame[0].header["WCDTSTAT"].lower() == "open" or frame[0].header["WCDTSTAT"].lower() == "idle"):

            if np.median(frame[0].data) < lampoff_threshold:
                print(f'[Info]: LAMPOFF FRAME filename={filenames[i]} object={frame[0].header["OBJECT"]} targname={frame[0].header["TARGNAME"]} altaz_distance={altaz_distance}')
                flats_lampoff.append(filenames[i])
            else:
                print(f'[Info]: LAMPON FRAME filename={filenames[i]} object={frame[0].header["OBJECT"]} targname={frame[0].header["TARGNAME"]} altaz_distance={altaz_distance}')
                flats_lampon.append(filenames[i])

        elif "sky" in frame[0].header["OBJECT"].lower() or "twi" in frame[0].header["OBJECT"].lower():

            print(f'[Info]: SKY FLAT FRAME filename={filenames[i]} object={frame[0].header["OBJECT"]} targname={frame[0].header["TARGNAME"]}')
            flats_sky.append(filenames[i])

        elif frame[0].header["SHRNAME"] == "closed":

            print(f'[Info]: DARK FRAME filename={filenames[i]} object={frame[0].header["OBJECT"]} targname={frame[0].header["TARGNAME"]}')
            darks.append(filenames[i])
        
        else:
            print(f'SCI FRAME filename={filenames[i]} object={frame[0].header["OBJECT"]} targname={frame[0].header["TARGNAME"]}')
            sci.append(filenames[i])

    return sci, flats, flats_sky, flats_lampon, flats_lampoff, darks

## Make Observing Log

In [73]:
def make_obslog(data_folder:str, date, obslog_filepath, sci, flats, flats_sky, flats_lampon, flats_lampoff, darks):

    obslog = {
        "data_folder":data_folder,
        "date":date,
        "raw":{
            "sci":sci,
            "flats":flats,
            "flats_sky":flats_sky,
            "flats_lampon":flats_lampon,
            "flats_lampoff":flats_lampoff,
            "darks":darks
        }
    }

    toml_string = toml.dumps(obslog)
    output_file = Path(obslog_filepath)
    
    with open(output_file, "w") as toml_file:
        toml.dump(obslog, toml_file)

Start with every single fits file in a directory like `../YYYY-MM-DD/data/*.fits`

In [ ]:
def generate_obslogs_generic():

    date = '2025-10-06'
    observation_folder = Path.cwd().parent / Path(date)

    output_folder = observation_folder / Path('obslogs')
    Path(output_folder).mkdir(exist_ok=True)

    frames = []
    filenames = []

    for filename in (Path(observation_folder) / Path('data')).glob('*.fits'):
        frames.append(fits.open(filename))
        filenames.append(Path(observation_folder) / Path('data') / Path(filename))

    sorted_frames = sort_frames(frames, filenames, filter_names="pol_cal")
    print(len(sorted_frames))
    make_obslog(output_folder, date, Path(output_folder) / Path(f'${date}_obslog.toml'), sorted_frames[0], sorted_frames[1], sorted_frames[2], sorted_frames[3], sorted_frames[4], sorted_frames[5])

    for frame in frames:
        frame.close()

In [105]:
generate_obslogs_generic()

[Info]: DARK FRAME filename=/home/shared/exoserver/NIRC2_Pol/ashish_reduction/2025-10-06/data/N2.20251007.08112.30.fits object=ao_confirmation targname=unknown
[Info]: DARK FRAME filename=/home/shared/exoserver/NIRC2_Pol/ashish_reduction/2025-10-06/data/N2.20251007.08225.52.fits object=test_script_run_hwp_0.0 targname=unknown
[Info]: DARK FRAME filename=/home/shared/exoserver/NIRC2_Pol/ashish_reduction/2025-10-06/data/N2.20251007.08241.33.fits object=test_script_run_hwp_45.0 targname=unknown
[Info]: DARK FRAME filename=/home/shared/exoserver/NIRC2_Pol/ashish_reduction/2025-10-06/data/N2.20251007.08256.63.fits object=test_script_run_hwp_22.5 targname=unknown
[Info]: DARK FRAME filename=/home/shared/exoserver/NIRC2_Pol/ashish_reduction/2025-10-06/data/N2.20251007.08273.46.fits object=test_script_run_hwp_67.5 targname=unknown
[Info]: DARK FRAME filename=/home/shared/exoserver/NIRC2_Pol/ashish_reduction/2025-10-06/data/N2.20251007.08448.39.fits object=k_flat targname=unknown
SCI FRAME file

## Bad Pixel Replacer

In [77]:
def median(data:np.array, mask:float, median_size, fail_val:float=0.0):
    '''
    Replaces bad pixels with the median of a box around them.
    '''
    bad_indicies = np.array(np.where(data < mask)).T
    half_size = np.floor(median_size / 2)

    h, w = data.shape
    h_i = h - 1
    w_i = w - 1
    
    for index in bad_indicies:

        i, j = index
        min_i = int(max(0, i-half_size))
        max_i = int(min(i+half_size, w_i) + 1)
        min_j = int(max(0, j-half_size))
        max_j = int(min(j+half_size, h_i) + 1)

        # if no good pixels, use fail_val
        if np.all(data[min_i:max_i, min_j:max_j] < mask):
            data[i,j] = fail_val
        else:
            data[i,j] = np.median(data[min_i:max_i, min_j:max_j])

## Make Darks

In [78]:
def make_sigma_clip_mask(image_data:np.array, n_sigma:float=9.0):

    if not image_data:
        return np.array([])

    mean_val = np.median(image_data)
    std_dev = np.std(image_data)
    threshold = mean_val + (n_sigma * std_dev)
    mask = image_data > threshold

    return mask

In [79]:
_mask_dir = Path.cwd().parent / Path('masks')
maskfile = Path(_mask_dir) / Path('bad_pixel_mask_20230101.fits')
NIRC2_bad_pixel_mask = fits.open(maskfile)            # from AstroImages or FITSIO

In [82]:
def make_masters(frames, keylist, n_sigma:float=6.0, median_size:int=7, method:callable=median, min_frames:int=3):

    frame_dict = dict(zip(keylist, frames))

    for key in list(frame_dict.keys()):
        if len(frame_dict[key]) < min_frames:
            print(f'[Warning]: Not enough frames for key {key}, skipping...')
            del(frame_dict[key])
        else:
            print(f'[Info]: Making master for key -> {key}, count -> {(len(frame_dict[key]))}')

    master_frames = {}
    master_frames_masks = {}

    for key in frame_dict.keys():

        if len(frame_dict[key]) < min_frames:
            print(f'[Warning]: Not enough frames for key {key}, skipping...')
            continue

        # applies method (mean, median) to frame list
        stack = np.array(frame_dict[key])
        method_stack = np.median(stack, axis=0)

        # take each frame, make a sigma clip mask, and then combine them or-wise

        masks = [make_sigma_clip_mask(frame, n_sigma) for frame in frame_dict[key]]
        sigma_clip_mask = np.any(mask, axis=0)

        # crop the original bad pixel mask to the size of the median frame
        bad_pixel_mask = NIRC2_bad_pixel_mask
        # if bad_pixel_mask.shape != method_stack.shape
        #     bad_pixel_mask, _, _ = crop(NIRC2_bad_pixel_mask, size(mf))
        # end

        # combine masks
        mask = bad_pixel_mask + sigma_clip_mask

        # make the median frame to do pixel replacement
        # not super efficient, but it works
        # median_mf = mapwindow(median, method_stack, (median_size, median_size))

        # # finally, assign the median values to the masked pixels
        # mf.data[mask] .= median_mf[mask]

        # repopulate the header with the key values so we can find the keys later
        for i, k in enumerate(keylist):
            method_stack[k] = key[i]

        method_stack["FILENAME"] = frames[1]["FILENAME"] # copy the first file name to the master
        method_stack["NFRAMES"] = len(frame_dict[key])
        method_stack["NSIGMASK"] = n_sigma
        method_stack["MEDSIZE"] = median_size
        method_stack["NPIXMASK"] = sum(mask)
        method_stack["MAMEDIAN"] = np.median(method_stack.data[not mask])
        method_stack["MAMEAN"] = np.mean(method_stack.data[not mask])
        method_stack["MASTD"] = np.std(method_stack.data[not mask])

        master_frames[key] = method_stack
        master_frames_masks[key] = mask

    return master_frames, master_frames_masks

In [83]:
def make_darks(darks_frames, darks_keylist=["NAXIS1", "NAXIS2", "ITIME", "COADDS"]):

    if darks_frames.shape != (0,):
        master_darks, master_darks_masks = make_masters(darks_frames, darks_keylist, method=median)
    else:
        master_darks = {}
        master_darks_masks = {}

    return master_darks, master_darks_masks,

In [84]:
def make_master_darks_obslog(obslog:dict, darks_keylist=["NAXIS1", "NAXIS2", "ITIME", "COADDS"]):
    darks_frames = obslog["darks"]
    master_darks, master_darks_masks = make_darks(darks_frames, darks_keylist=darks_keylist)
    return master_darks, master_darks_masks

In [85]:
def all_header_keywords_match(ha, hb, kws):
    for kw in kws:
        if ha[kw] != hb[kw]:
            return False
    return True

In [86]:
def find_matching_master(frame, masters, keylist):

    if masters == None:
        return None, None

    matches = [all_header_keywords_match(frame, m, keylist) for m in masters]

    if np.any(matches):
        ind = np.argmax(matches)
        matched = masters[ind]
    else:
        return None, None

    return ind, matched

In [88]:
def find_closest_dark(frame, master_darks, ranked_darks_keylist=[["NAXIS1", "NAXIS2", "ITIME", "COADDS", "SAMPMODE", "READS"], ["NAXIS1", "NAXIS2", "ITIME", "COADDS", "SAMPMODE"], ["NAXIS1", "NAXIS2", "ITIME", "COADDS"], ["NAXIS1", "NAXIS2", "ITIME"], ["ITIME"]]):
    """
    Guarantees finding a dark frame that matches at least the ITIME. Frame should be cropped as well.

    # Arguments
    - `frame`: The frame for which we want to find a matching dark.
    - `master_darks`: A list of master dark frames.
    - `ranked_darks_keylist`: A list of keylists in order of preference for matching dark frames. Each keylist is a list of header keywords that should match. At minimum we want to match the ITIME.
    """
    
    # if master_darks == AstroImage[]
    #     return nothing, nothing

    matched_dark = None
    ind = None
    
    for i, keylist in enumerate(ranked_darks_keylist):
        ind, matched_dark = find_matching_master(frame, master_darks, keylist)
        if matched_dark != None:

            # for the 2nd and 3rd case, rescale by coadds
            if (i == 4) or (i == 5):
                print(f'[Warning]: Rescaling dark frame by COADDS {matched_dark["COADDS"]} -> {frame[0].header["COADDS"]}')
                matched_dark = matched_dark / matched_dark["COADDS"] * frame[0].header["COADDS"]

            if i == 5:
                if matched_dark.shape != frame.shape:
                    if np.all(np.array(matched_dark.shape) > np.array(frame.shape)):
                        cropped_dark = matched_dark[:frame.shape[0], :frame.shape[1]]
                        #matched_dark = AstroImage(cropped_dark, matched_dark.header)
                    else:
                        print(f'[Warning]: Dark frame is smaller than the target frame, not cropping: {frame[0].header["FILENAME"]}')
                        return None
            break

    if matched_dark == None:
        print(f'[Warning]: No matching dark found for {frame[0].header["FILENAME"]}, {frame[0].header["FILTER"]}, {frame[0].header["ITIME"]}, {frame[0].header["COADDS"]}')

    return ind, matched_dark


In [89]:
def make_flats(flat_frames, master_darks, flats_keylist=["NAXIS1", "NAXIS2", "ITIME", "COADDS", "FILTER"], flattype="REGULAR"):
    if flat_frames.shape != (0,):
        master_flats, master_flats_masks = make_masters(flat_frames, flats_keylist, method=median)

        for key in master_flats.keys():
            
            matched_dark = find_closest_dark(master_flats[key], master_darks)

            if matched_dark != None:
                print('[Info]: Subtracting dark from master flat')
                master_flats[key] -= matched_dark
                master_flats[key]["FLATTYPE"] = flattype
            else:
                print('[Warning]: No matching dark found for master flat')
                master_flats[key]["FLATTYPE"] = "$(flattype)+NODARK"

            master_flats[key] /= master_flats[key]["MAMEDIAN"]

    else:
        print('[Warning]: No regular m[:frflats found, skipping...')
        master_flats = {}
        master_flats_masks = {}

    return master_flats, master_flats_masks